In [ ]:
import os 

os.chdir("..")

In [ ]:
import polars as pl
from fsdc_calories import DataCal
import plotly.graph_objects as go
from importlib import resources
from plotly.subplots import make_subplots
import pandas as pd 
dc = DataCal()

In [ ]:
dc.gen_graphs_price_change()

In [ ]:
df = dc.process_price()
df

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = dc.process_price().to_pandas()
df['pct_change_imports_year_over_year'] = df[
    'pct_change_imports_year_over_year'
]  # .clip(upper=500, lower=-500)

# Create a Year-Month tracking column
df['year_month'] = (
    df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
)
last_5_months = sorted(df['year_month'].unique())[-5:]

# 2. Create side-by-side subplots (1 row, 2 columns)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        'Top 10 Import Increases (% Change)',
        'Bottom 10 Import Decreases (% Change)',
    ),
    horizontal_spacing=0.25,
)

traces_per_month_count = (
    20  # 10 bars for top chart + 10 bars for bottom chart = 20 traces per month
)
total_months = len(last_5_months)

# 3. Loop through the last 5 months to construct traces
for ym in last_5_months:
  sub_df = df[df['year_month'] == ym]

  # Aggregate mean percent change by hs4
  agg = (
      sub_df.groupby('hs4')['pct_change_imports_year_over_year']
      .mean()
      .reset_index()
  )

  # Get Top 10 and sort ascending so the highest value ends up at the top
  top_10 = agg.nlargest(10, 'pct_change_imports_year_over_year').sort_values(
      by='pct_change_imports_year_over_year', ascending=True
  )

  # Get Bottom 10 and sort ascending
  bottom_10 = agg.nsmallest(10, 'pct_change_imports_year_over_year').sort_values(
      by='pct_change_imports_year_over_year', ascending=True
  )

  # Add Top 10 traces (Left Column - Col 1)
  for _, row in top_10.iterrows():
    hs = row['hs4']
    val = row['pct_change_imports_year_over_year']
    color = '#2ca02c' if val >= 0 else '#d62728'  # Green for increase, Red

    fig.add_trace(
        go.Bar(
            x=[val],
            y=[str(hs)],
            orientation='h',
            marker_color=color,
            visible=False,
            showlegend=False,
            hovertemplate=(
                '<b>HS4:</b> %{y}<br><b>Change:</b> %{x:.2f}%<extra></extra>'
            ),
        ),
        row=1,
        col=1,
    )

  # Add Bottom 10 traces (Right Column - Col 2)
  for _, row in bottom_10.iterrows():
    hs = row['hs4']
    val = row['pct_change_imports_year_over_year']
    color = '#2ca02c' if val >= 0 else '#d62728'

    fig.add_trace(
        go.Bar(
            x=[val],
            y=[str(hs)],
            orientation='h',
            marker_color=color,
            visible=False,
            showlegend=False,
            hovertemplate=(
                '<b>HS4:</b> %{y}<br><b>Change:</b> %{x:.2f}%<extra></extra>'
            ),
        ),
        row=1,
        col=2,
    )

# 4. Build Dropdown Menu Buttons for the last 5 months
buttons = []
for i, ym in enumerate(last_5_months):
  visibility = [False] * (total_months * traces_per_month_count)
  start_idx = i * traces_per_month_count
  end_idx = start_idx + traces_per_month_count

  for j in range(start_idx, end_idx):
    visibility[j] = True

  buttons.append(
      dict(
          label=ym,
          method='update',
          args=[
              {'visible': visibility},
              {
                  'title': (
                      'Top 10 Increases & Bottom 10 Decreases in Import Value'
                      f' Change ({ym}) [Capped at ±500%]'
                  )
              },
          ],
      )
  )

# Set the most recent month visible by default
for j in range(
    (total_months - 1) * traces_per_month_count,
    total_months * traces_per_month_count,
):
  fig.data[j].visible = True

# 5. Global Layout & Reference Lines Configuration
fig.update_layout(
    updatemenus=[
        dict(
            active=total_months - 1,
            buttons=buttons,
            x=0.0,
            y=1.18,
            xanchor='left',
            yanchor='top',
            showactive=True,
        )
    ],
    height=550,
    width=1000,
    title_text=(
        'Top 10 Increases & Bottom 10 Decreases in Import Value Change'
        f' ({last_5_months[-1]}) [Capped at ±500%]'
    ),
    template='plotly_white',
)

# Add dashed zero-reference lines to both subplots for visual separation
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)

# Axis labels
fig.update_xaxes(title_text='Percentage Change (%)', row=1, col=1)
fig.update_xaxes(title_text='Percentage Change (%)', row=1, col=2)

# FIX: Add categoryorder='trace' so Plotly follows the dataframe's sort order instead of sorting alphabetically
fig.update_yaxes(
    title_text='HS4 Category', categoryorder='trace', row=1, col=1
)
fig.update_yaxes(
    title_text='HS4 Category', categoryorder='trace', row=1, col=2
)

# Display the chart
fig.show()

In [ ]:
# 1. Load description lookup table using your resource path
df = dc.process_price().to_pandas()
path_data = str(
    resources.files('fsdc_calories').joinpath('data/hts4_walk.parquet')
)
desc_df = (
    pl.read_parquet(path_data)
    .rename({'code_4': 'hs4', 'Description': 'hts_desc'})
    .to_pandas()
)
desc_df = desc_df.drop_duplicates(subset=['hs4'])  # prevent duplicate mapping

# 2. Fetch ranking data & apply clipping
df['pct_change_imports_year_over_year'] = df[
    'pct_change_imports_year_over_year'
].clip(upper=500, lower=-500)

# Create Year-Month column for filtering the last 5 months
df['year_month'] = (
    df['year'].astype(str)
    + '-'
    + df['month'].astype(str).str.zfill(2)
)
last_5_months = sorted(df['year_month'].unique())[-5:]

# 3. Initialize side-by-side subplots (1 row, 2 columns)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        'Top 10 Import Increases (% Change)',
        'Bottom 10 Import Decreases (% Change)',
    ),
    horizontal_spacing=0.35,  # Extra spacing for description labels on y-axis
)

traces_per_month_count = (
    20  # 10 bars for top chart + 10 bars for bottom chart = 20 traces per month
)
total_months = len(last_5_months)

# 4. Loop through the last 5 months to construct traces
for ym in last_5_months:
  sub_df = df[df['year_month'] == ym]

  # Aggregate mean percent change by hs4
  agg = (
      sub_df.groupby('hs4')['pct_change_imports_year_over_year']
      .mean()
      .reset_index()
  )

  # Merge with descriptions
  agg_merged = agg.merge(desc_df, on='hs4', how='left')
  agg_merged['hts_desc'] = agg_merged['hts_desc'].fillna(
      'Unknown Commodity'
  )  # fallback label

  # Get Top 10 and sort ascending for proper horizontal bar layout
  top_10 = agg_merged.nlargest(
      10, 'pct_change_imports_year_over_year'
  ).sort_values(by='pct_change_imports_year_over_year', ascending=True)

  # Get Bottom 10 and sort ascending
  bottom_10 = agg_merged.nsmallest(
      10, 'pct_change_imports_year_over_year'
  ).sort_values(by='pct_change_imports_year_over_year', ascending=True)

  # Add Top 10 traces (Left Column - Col 1) using truncated description for y-axis
  for _, row in top_10.iterrows():
    hs = row['hs4']
    full_desc = str(row['hts_desc'])
    desc_short = (
        full_desc[:97] + '...' if len(full_desc) > 100 else full_desc
    )
    val = row['pct_change_imports_year_over_year']
    color = '#2ca02c' if val >= 0 else '#d62728'

    fig.add_trace(
        go.Bar(
            x=[val],
            y=[desc_short],
            orientation='h',
            marker_color=color,
            visible=False,
            showlegend=False,
            hovertemplate=(
                f'<b>HS4:</b> {hs}<br><b>Description:</b> {full_desc}<br><b>Change:'
                '</b> %{x:.2f}%<extra></extra>'
            ),
        ),
        row=1,
        col=1,
    )

  # Add Bottom 10 traces (Right Column - Col 2) using truncated description for y-axis
  for _, row in bottom_10.iterrows():
    hs = row['hs4']
    full_desc = str(row['hts_desc'])
    desc_short = (
        full_desc[:97] + '...' if len(full_desc) > 100 else full_desc
    )
    val = row['pct_change_imports_year_over_year']
    color = '#2ca02c' if val >= 0 else '#d62728'

    fig.add_trace(
        go.Bar(
            x=[val],
            y=[desc_short],
            orientation='h',
            marker_color=color,
            visible=False,
            showlegend=False,
            hovertemplate=(
                f'<b>HS4:</b> {hs}<br><b>Description:</b> {full_desc}<br><b>Change:'
                '</b> %{x:.2f}%<extra></extra>'
            ),
        ),
        row=1,
        col=2,
    )

# 5. Build Dropdown Menu Buttons for the last 5 months
buttons = []
for i, ym in enumerate(last_5_months):
  visibility = [False] * (total_months * traces_per_month_count)
  start_idx = i * traces_per_month_count
  end_idx = start_idx + traces_per_month_count

  for j in range(start_idx, end_idx):
    visibility[j] = True

  buttons.append(
      dict(
          label=ym,
          method='update',
          args=[
              {'visible': visibility},
              {
                  'title': (
                      'Top 10 Increases & Bottom 10 Decreases in Import Value'
                      f' Change ({ym}) [Capped at ±500%]'
                  )
              },
          ],
      )
  )

# Set the most recent month visible by default
for j in range(
    (total_months - 1) * traces_per_month_count,
    total_months * traces_per_month_count,
):
  fig.data[j].visible = True

# 6. Global Layout & Reference Lines Configuration
fig.update_layout(
    updatemenus=[
        dict(
            active=total_months - 1,
            buttons=buttons,
            x=0.0,
            y=1.18,
            xanchor='left',
            yanchor='top',
            showactive=True,
        )
    ],
    height=600,
    width=1200,
    title_text=(
        'Top 10 Increases & Bottom 10 Decreases in Import Value Change'
        f' ({last_5_months[-1]}) [Capped at ±500%]'
    ),
    template='plotly_white',
    margin=dict(
        l=250
    ),  # Extra left margin to fit full length description labels nicely
)

# Add dashed zero-reference lines to both subplots for visual separation
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)

# Axis labels & configuration
fig.update_xaxes(title_text='Percentage Change (%)', row=1, col=1)
fig.update_xaxes(title_text='Percentage Change (%)', row=1, col=2)
fig.update_yaxes(title_text='', row=1, col=1)
fig.update_yaxes(title_text='', row=1, col=2)

# Display the chart
fig.show()

In [ ]:
dc.gen_graphs_plate()